In [1]:
import sys
import sysconfig

print(sys.executable)
print(sysconfig.get_platform())
if sysconfig.get_platform().startswith("mingw"):
    raise RuntimeError("Select the project .venv/Scripts/python.exe kernel instead of MSYS2 Python.")

%pip install pyserial ipywidgets


a:\laragon\bin\python\python-3.13\python.exe
win-amd64
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import queue
import threading
import time

import ipywidgets as widgets
import serial
from IPython.display import display
from serial.tools import list_ports

for name in ("_disconnect_motor", "_disconnect_rpm"):
    if name in globals():
        globals()[name]()

def control_loop(connection, targets, stopped, on_line=None, command_prefix="PWM:"):
    """One thread owns serial I/O, including the heartbeat and final stop."""
    report = on_line if on_line is not None else lambda line: print(f"\n{line}", flush=True)
    target = 0
    next_send = 0.0
    try:
        while not stopped.is_set():
            try:
                while True:
                    target = targets.get_nowait()
                    next_send = 0.0
            except queue.Empty:
                pass

            now = time.monotonic()
            if now >= next_send:
                connection.write(f"{command_prefix}{target}\n".encode("ascii"))
                next_send = now + 0.5

            line = connection.readline().decode("ascii", errors="replace").strip()
            if line:
                report(line)
    except (serial.SerialException, OSError) as error:
        report(f"Connection lost: {error}")
    finally:
        stopped.set()
        try:
            connection.write(f"{command_prefix}0\n".encode("ascii"))
            connection.flush()
        except (serial.SerialException, OSError):
            pass


def create_controls():
    port = widgets.Combobox(description="USB port:", placeholder="COM4")
    refresh = widgets.Button(description="Refresh ports")
    connect = widgets.Button(description="Connect", button_style="success")
    disconnect = widgets.Button(description="Disconnect", disabled=True)
    pwm = widgets.BoundedIntText(
        description="Target PWM:", value=0, min=0, max=255, disabled=True
    )
    apply = widgets.Button(description="Apply PWM", disabled=True)
    on = widgets.Button(description="ON", button_style="success", disabled=True)
    up = widgets.Button(description="Speed +10", disabled=True)
    down = widgets.Button(description="Speed -10", disabled=True)
    stop = widgets.Button(description="STOP", button_style="danger", disabled=True)
    status = widgets.Label(value="Disconnected")
    output = widgets.Label(value="Board PWM: unavailable")
    led = widgets.Label(value="LED drive indicator: unavailable")
    fault = widgets.Label(value="Board fault: unavailable")
    message = widgets.Label(value="Choose the USB port used in Arduino IDE.")
    mode = widgets.Label(value="PWM speed control (0-255). Actual RPM is not measured.")
    state = {
        "worker": None, "targets": queue.Queue(),
        "stopped": threading.Event(), "requested": 0,
    }
    run_controls = (pwm, apply, on, up, down)

    def set_connected(value):
        port.disabled = refresh.disabled = connect.disabled = value
        disconnect.disabled = stop.disabled = not value
        for control in run_controls:
            control.disabled = True 

    def show_line(line):
        message.value = line
        fields = line.split(",")
        if fields[0] == "STATE" and (len(fields) != 5 or fields[1] != "PWM"):
            status.value = "Serial reply received; upload the rewritten PWM sketch first"
            return
        if len(fields) != 5 or fields[:2] != ["STATE", "PWM"]:
            return
        try:
            duty, led_value = int(fields[2]), int(fields[3])
        except ValueError:
            return
        if not 0 <= duty <= 255 or led_value not in (0, 1):
            return
        output.value = f"Board PWM: {duty}/255 ({100 * duty / 255:.1f}%)"
        led.value = f"LED drive indicator: {'ON' if led_value else 'OFF'}"
        fault.value = f"Board fault: {fields[4]}"
        status.value = f"Board responding (PWM) on {port.value.strip()}"
        for control in run_controls:
            control.disabled = fields[4] != "NONE"
        if fields[4] == "COMMAND_TIMEOUT":
            message.value = "Heartbeat timed out. Click STOP to acknowledge, then choose a speed."
        elif fields[4] == "PWM_SETUP":
            message.value = "PWM initialization failed; check the selected ESP32 board and sketch."

    def refresh_ports(_=None):
        ports = list(list_ports.comports())
        port.options = tuple(item.device for item in ports)
        if not port.value:
            preferred = next((
                item.device for item in ports
                if any(word in (getattr(item, "description", "") or "").lower()
                       for word in ("cp210", "ch340", "ch341", "esp32", "usb serial"))
            ), None)
            port.value = preferred or (port.options[0] if len(port.options) == 1 else "")

    def connect_board(_):
        connection = None
        try:
            if not port.value.strip():
                raise ValueError("Select or type a USB port, such as COM4.")
            connection = serial.Serial(
                port.value.strip(), 115200, timeout=0.1, write_timeout=1
            )
            time.sleep(2)  
        except (serial.SerialException, OSError, ValueError) as error:
            if connection is not None:
                connection.close()
            message.value = f"Cannot connect: {error}. Close Arduino Serial Monitor."
            return

        state["targets"] = queue.Queue()
        state["stopped"] = threading.Event()
        state["requested"] = pwm.value = 0
        output.value = "Board PWM: waiting for board"
        led.value = "LED drive indicator: unavailable"
        fault.value = "Board fault: waiting for board"
        status.value = f"USB port {port.value.strip()} open; waiting for PWM sketch"
        set_connected(True)

        def run():
            try:
                control_loop(
                    connection, state["targets"], state["stopped"],
                    on_line=show_line, command_prefix="PWM:"
                )
            finally:
                connection.close()
                set_connected(False)
                status.value = "Disconnected; displayed readings are from the last connection."

        state["worker"] = threading.Thread(target=run, daemon=True)
        state["worker"].start()

    def send_pwm(value):
        value = max(0, min(255, int(value)))
        state["requested"] = pwm.value = value
        state["targets"].put(value)
        message.value = f"Requested PWM {value}/255; check Board PWM for confirmation."

    def stop_motor(_=None):
        send_pwm(0)

    def disconnect_board(_=None):
        state["stopped"].set()
        worker = state["worker"]
        if worker is not None:
            worker.join(timeout=3)
            if worker.is_alive():
                raise RuntimeError("Serial shutdown is still pending. Retry Disconnect.")
        status.value = "Disconnected"
        set_connected(False)

    refresh.on_click(refresh_ports)
    connect.on_click(connect_board)
    disconnect.on_click(disconnect_board)
    apply.on_click(lambda _: send_pwm(pwm.value))
    on.on_click(lambda _: send_pwm(pwm.value if pwm.value > 0 else 200))
    up.on_click(lambda _: send_pwm(state["requested"] + 10))
    down.on_click(lambda _: send_pwm(state["requested"] - 10))
    stop.on_click(stop_motor)
    refresh_ports()

    panel = widgets.VBox([
        widgets.HBox([port, refresh, connect, disconnect]),
        widgets.HBox([pwm, apply]),
        widgets.HBox([on, up, down, stop]),
        status, mode, output, led, fault, message,
    ])
    return panel, disconnect_board


motor_panel, _disconnect_motor = create_controls()
display(motor_panel)
